# ComplaintRouter — Data Cleaning & Sampling

This notebook prepares CFPB consumer complaint narrative data for
ComplaintRouter's product classification task.

## 1. Load CFPB Archive Files

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
data_path = Path("../data/raw")
csv_files = sorted(data_path.glob("*.csv"))
print(f"Found {len(csv_files)} csv files")
for file in csv_files:
    print(file.name)

Found 17 csv files
CCDB_Export_10_March_2025_through_April_2025.csv
CCDB_Export_11_May_2025_through_June_2025.csv
CCDB_Export_12_July_2025_through_August_2025.csv
CCDB_Export_13_September_2025_through_October_2025.csv
CCDB_Export_14_November_2025_through_December_2025.csv
CCDB_Export_15_January_2026_through_February_2026.csv
CCDB_Export_16_March_2026.csv
CCDB_Export_17_April_2026.csv
CCDB_Export_18_May_2026.csv
CCDB_Export_19_June_2026.csv
CCDB_Export_20_July_2026.csv
CCDB_Export_21_August_2026.csv
CCDB_Export_5_September_2023_through_March_2024.csv
CCDB_Export_6_April_2024_through_July_2024.csv
CCDB_Export_7_August_2024_through_October_2024.csv
CCDB_Export_8_November_2024_through_December_2024.csv
CCDB_Export_9_January_2025_through_February_2025.csv


In [3]:
sample_df = pd.read_csv(csv_files[0])

print("Shape:", sample_df.shape)
print("\nColumns:")
print(sample_df.columns.tolist())

sample_df.head()

Shape: (803073, 16)

Columns:
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID']


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2025-03-06,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,33016,NaN,Web,2025-03-06,Closed with explanation,Yes,12357992
1,2025-03-06,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",FL,33016,NaN,Web,2025-03-06,Closed with non-monetary relief,Yes,12357594
2,2025-03-06,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"EQUIFAX, INC.",FL,33016,NaN,Web,2025-03-06,Closed with non-monetary relief,Yes,12354528
3,2025-03-20,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,34744,NaN,Web,2025-03-20,Closed with explanation,Yes,12553463
4,2025-03-20,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",FL,34744,NaN,Web,2025-03-20,Closed with non-monetary relief,Yes,12553457


## 2. Filter Complaints With Narratives

In [4]:
print("Total complaints:", len(sample_df))

print(
    "Complaints with narratives:",
    sample_df["Consumer complaint narrative"].notna().sum()
)

print(
    "Percent with narratives:",
    round(
        sample_df["Consumer complaint narrative"].notna().mean() * 100,
        2
    ),
    "%"
)

print("\nDate range:")
print(sample_df["Date received"].min())
print(sample_df["Date received"].max())

print("\nProducts:")
print(sample_df["Product"].value_counts())

Total complaints: 803073
Complaints with narratives: 210147
Percent with narratives: 26.17 %

Date range:
2025-03-01
2025-04-30

Products:
Product
Credit reporting or other personal consumer reports        718052
Debt collection                                             40673
Credit card                                                 13152
Checking or savings account                                 10299
Student loan                                                 5572
Money transfer, virtual currency, or money service           5536
Mortgage                                                     3670
Vehicle loan or lease                                        2732
Payday loan, title loan, personal loan, or advance loan      1754
Prepaid card                                                  993
Debt or credit management                                     640
Name: count, dtype: int64


In [5]:
columns_to_keep = [
    "Date received",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Consumer complaint narrative",
    "Company",
    "Company response to consumer",
    "Timely response?",
    "Complaint ID"
]

narrative_dfs = []

for file in csv_files:
    df = pd.read_csv(file, usecols=columns_to_keep)

    # Keep only complaints that contain a narrative
    df = df.dropna(subset=["Consumer complaint narrative"])

    narrative_dfs.append(df)

    print(f"{file.name}: {len(df):,} narratives")

CCDB_Export_10_March_2025_through_April_2025.csv: 210,147 narratives
CCDB_Export_11_May_2025_through_June_2025.csv: 219,632 narratives
CCDB_Export_12_July_2025_through_August_2025.csv: 234,404 narratives
CCDB_Export_13_September_2025_through_October_2025.csv: 157,821 narratives
CCDB_Export_14_November_2025_through_December_2025.csv: 134,641 narratives
CCDB_Export_15_January_2026_through_February_2026.csv: 34,351 narratives
CCDB_Export_16_March_2026.csv: 21,640 narratives
CCDB_Export_17_April_2026.csv: 21,052 narratives
CCDB_Export_18_May_2026.csv: 17,528 narratives


/var/folders/pg/90vjsjd154z3t7xnyft_tbyw0000gn/T/ipykernel_16558/3712166456.py:17: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, usecols=columns_to_keep)


CCDB_Export_19_June_2026.csv: 16,820 narratives


/var/folders/pg/90vjsjd154z3t7xnyft_tbyw0000gn/T/ipykernel_16558/3712166456.py:17: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, usecols=columns_to_keep)


CCDB_Export_20_July_2026.csv: 10,002 narratives


/var/folders/pg/90vjsjd154z3t7xnyft_tbyw0000gn/T/ipykernel_16558/3712166456.py:17: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, usecols=columns_to_keep)


CCDB_Export_21_August_2026.csv: 1 narratives
CCDB_Export_5_September_2023_through_March_2024.csv: 318,804 narratives
CCDB_Export_6_April_2024_through_July_2024.csv: 276,332 narratives
CCDB_Export_7_August_2024_through_October_2024.csv: 229,944 narratives
CCDB_Export_8_November_2024_through_December_2024.csv: 154,916 narratives
CCDB_Export_9_January_2025_through_February_2025.csv: 265,405 narratives


In [6]:
complaints = pd.concat(narrative_dfs, ignore_index=True)

print("Combined shape:", complaints.shape)
print("Total narratives:", f"{len(complaints):,}")

complaints.head()

Combined shape: (2323440, 10)
Total narratives: 2,323,440


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company,Company response to consumer,Timely response?,Complaint ID
0,2025-03-24,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,I DEMAND THAT THESE FRAUDULENT HARD INQUIRIES ...,"EQUIFAX, INC.",Closed with explanation,Yes,12623706
1,2025-03-24,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,I DEMAND THAT THESE FRAUDULENT HARD INQUIRIES ...,Experian Information Solutions Inc.,Closed with explanation,Yes,12648214
2,2025-03-24,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,I DEMAND THAT THESE FRAUDULENT HARD INQUIRIES ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",Closed with non-monetary relief,Yes,12638196
3,2025-03-24,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account status incorrect,I REQUEST THE IMMEDIATE REMOVAL OF THIS LATE P...,"EQUIFAX, INC.",Closed with non-monetary relief,Yes,12646738
4,2025-03-24,Debt collection,I do not know,Attempts to collect debt not owed,Debt is not yours,PLEASE BLOCKED AND REMOVED I respectfully requ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",Closed with explanation,Yes,12642342


## 3. Investigate Duplicate Narratives

In [7]:
print("Total rows:", f"{len(complaints):,}")

print(
    "Duplicate Complaint IDs:",
    f"{complaints['Complaint ID'].duplicated().sum():,}"
)

print(
    "Duplicate narratives:",
    f"{complaints['Consumer complaint narrative'].duplicated().sum():,}"
)

Total rows: 2,323,440
Duplicate Complaint IDs: 0
Duplicate narratives: 1,007,818


In [8]:
duplicate_counts = (
    complaints["Consumer complaint narrative"]
    .value_counts()
)

print("Unique narratives:", f"{complaints['Consumer complaint narrative'].nunique():,}")
print("\nMost repeated narratives:")
print(duplicate_counts.head(10))

Unique narratives: 1,315,622

Most repeated narratives:
Consumer complaint narrative
In accordance with the Fair Credit Reporting act. The List of accounts below has violated my federally protected consumer rights to privacy and confidentiality under 15 USC 1681.\n\n15 U.S.C 1681 section 602 A. States I have the right to privacy.\n\n15 U.S.C 1681 Section 604 A Section 2 : It also states a consumer reporting agency can not furnish a account without my written instructions 15 U.S.C 1681c. ( a ) ( 5 ) Section States : no consumer reporting agency may make any consumer report containing any of the following items of information Any other adverse item of information, other than records of convictions of crimes which antedates the report by more than seven years.\n\n15 U.S.C. 1681s-2 ( A ) ( 1 ) A person shall not furnish any information relating to a consumer to any consumer reporting agency if the person knows or has reasonable cause to believe that the information is inaccurate.          

In [9]:
narrative_product_counts = (
    complaints.groupby("Consumer complaint narrative")["Product"]
    .nunique()
)

print(
    "Narratives associated with multiple products:",
    f"{(narrative_product_counts > 1).sum():,}"
)

Narratives associated with multiple products: 1,650


## 4. Remove Duplicate and Ambiguous Narratives

In [10]:
clean_complaints = complaints.drop_duplicates(
    subset=["Consumer complaint narrative"],
    keep="first"
).copy()

print("Before deduplication:", f"{len(complaints):,}")
print("After deduplication:", f"{len(clean_complaints):,}")
print("Removed:", f"{len(complaints) - len(clean_complaints):,}")

Before deduplication: 2,323,440
After deduplication: 1,315,622
Removed: 1,007,818


## 5. Inspect Product Distribution

In [11]:
print("Missing values:")
print(clean_complaints.isnull().sum())

print("\nDate range:")
print(clean_complaints["Date received"].min())
print(clean_complaints["Date received"].max())

print("\nNumber of products:")
print(clean_complaints["Product"].nunique())

print("\nProduct distribution:")
print(clean_complaints["Product"].value_counts())

Missing values:
Date received                       0
Product                             0
Sub-product                         0
Issue                               0
Sub-issue                       92687
Consumer complaint narrative        0
Company                             0
Company response to consumer        3
Timely response?                    0
Complaint ID                        0
dtype: int64

Date range:
2023-09-01
2026-08-11

Number of products:
11

Product distribution:
Product
Credit reporting or other personal consumer reports        752276
Debt collection                                            166574
Checking or savings account                                106744
Credit card                                                100258
Money transfer, virtual currency, or money service          63475
Mortgage                                                    38002
Vehicle loan or lease                                       27219
Student loan                           

In [12]:
ambiguous_narratives = narrative_product_counts[
    narrative_product_counts > 1
].index

clean_complaints = complaints[
    ~complaints["Consumer complaint narrative"].isin(ambiguous_narratives)
].drop_duplicates(
    subset=["Consumer complaint narrative"]
).copy()

print("Final unique, unambiguous narratives:", f"{len(clean_complaints):,}")

Final unique, unambiguous narratives: 1,313,972


## 6. Create Modeling Sample

In [13]:
sample_sizes = {
    "Credit reporting or other personal consumer reports": 35000,
    "Debt collection": 25000,
    "Checking or savings account": 20000,
    "Credit card": 20000,
    "Money transfer, virtual currency, or money service": 15000,
    "Mortgage": 10000,
    "Vehicle loan or lease": 10000,
    "Student loan": 10000,
    "Payday loan, title loan, personal loan, or advance loan": 7500,
    "Prepaid card": 5000,
    "Debt or credit management": 5000
}

In [14]:
sampled_dfs = []

for product, target_size in sample_sizes.items():
    product_data = clean_complaints[
        clean_complaints["Product"] == product
    ]

    # Never request more records than actually exist
    n = min(target_size, len(product_data))

    sampled = product_data.sample(
        n=n,
        random_state=42
    )

    sampled_dfs.append(sampled)

modeling_data = pd.concat(
    sampled_dfs,
    ignore_index=True
)

# Shuffle the combined dataset
modeling_data = modeling_data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Final modeling dataset:", f"{len(modeling_data):,}")
print("\nClass distribution:")
print(modeling_data["Product"].value_counts())

Final modeling dataset: 162,500

Class distribution:
Product
Credit reporting or other personal consumer reports        35000
Debt collection                                            25000
Credit card                                                20000
Checking or savings account                                20000
Money transfer, virtual currency, or money service         15000
Vehicle loan or lease                                      10000
Student loan                                               10000
Mortgage                                                   10000
Payday loan, title loan, personal loan, or advance loan     7500
Debt or credit management                                   5000
Prepaid card                                                5000
Name: count, dtype: int64


## 7. Save Processed Dataset

In [15]:
output_path = Path("../data/processed/complaint_router_modeling_data.csv")

modeling_data.to_csv(output_path, index=False)

print(f"Saved {len(modeling_data):,} rows to:")
print(output_path)

Saved 162,500 rows to:
../data/processed/complaint_router_modeling_data.csv


In [16]:
test_load = pd.read_csv(output_path)

print("Saved shape:", test_load.shape)
print("Missing narratives:", test_load["Consumer complaint narrative"].isna().sum())
print("Number of classes:", test_load["Product"].nunique())

Saved shape: (162500, 10)
Missing narratives: 0
Number of classes: 11
